In [ ]:
import numpy as np
import pandas as pd
import os

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

In [ ]:
DATASET_PATH_CSV = 'PCOS_infertility.csv'
DATASET_PATH_EXCEL = 'PCOS_data_without_infertility.xlsx'
assert os.path.isfile(DATASET_PATH_CSV),f'verifier que le path :{DATASET_PATH} est bien defini'
assert os.path.isfile(DATASET_PATH_EXCEL),f'verifier que le path :{DATASET_PATH} est bien defini'

data_csv = pd.read_csv(DATASET_PATH_CSV)
data_xsls = pd.read_excel(DATASET_PATH_EXCEL)

data_csv.head()

,Sl. No,Patient File No.,PCOS (Y/N),I beta-HCG(mIU/mL),II beta-HCG(mIU/mL),AMH(ng/mL)
0,1,10001,0,1.99,1.99,2.07
1,2,10002,0,60.80,1.99,1.53
2,3,10003,1,494.08,494.08,6.63
3,4,10004,0,1.99,1.99,1.22
4,5,10005,0,801.45,801.45,2.26


In [ ]:
print(f'CSV SHAPE:{data_csv.shape} - EXCEL SHAPE {data_xsls.shape}')

CSV SHAPE:(541, 6) - EXCEL SHAPE (20, 4)


In [ ]:
data_csv.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Sl. No                  541 non-null    int64  
 1   Patient File No.        541 non-null    int64  
 2   PCOS (Y/N)              541 non-null    int64  
 3     I   beta-HCG(mIU/mL)  541 non-null    float64
 4   II    beta-HCG(mIU/mL)  541 non-null    float64
 5   AMH(ng/mL)              541 non-null    object 
dtypes: float64(2), int64(3), object(1)
memory usage: 25.5+ KB


In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

from collections import Counter
#from mlxtend.plotting import plot_confusion_matrix
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost
#import lightgbm
#from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
columns =  data.columns
columns

Index(['Sl. No', 'Patient File No.', 'PCOS (Y/N)', '  I   beta-HCG(mIU/mL)',
       'II    beta-HCG(mIU/mL)', 'AMH(ng/mL)'],
      dtype='object')

In [1]:
# Jupyter-style script (VSCode interactive / Jupyter can run cells separated by "# %%")

# %% [markdown]
# Analyse visuelle PCOS — distribution des classes et graphiques exploratoires

# %%
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# tenter de réutiliser les utilitaires existants
try:
    from pcos_analysis import load_data, clean_columns
except Exception:
    # fallback: charger directement le CSV dans le même dossier
    DATA_PATH = os.path.join(os.path.dirname(__file__), "PCOS_infertility.csv")
    def load_data(path=DATA_PATH):
        return pd.read_csv(path)
    def clean_columns(df):
        return df.rename(columns=lambda c: c.strip())

# %% 
# Chargement et préparation légère
df = load_data()
df = clean_columns(df)

# convertir en numérique comme dans pcos_analysis
for col in df.columns:
    if col != 'target':
        df[col] = pd.to_numeric(df[col], errors='coerce')
df['target'] = pd.to_numeric(df['target'], errors='coerce').astype('Int64')

print("Shape:", df.shape)
display(df.head())

# %% [markdown]
# 1) Distribution de la cible — vérifier l'imbalancedness

# %%
# Distribution cible
vc = df['target'].value_counts(dropna=False)
pct = df['target'].value_counts(normalize=True) * 100
print("Counts:\n", vc.to_dict())
print("\nPercent:\n", pct.round(2).to_dict())

fig, ax = plt.subplots(1,2, figsize=(10,4))
sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax[0], palette='muted')
ax[0].set_title('Counts par classe (target)')
ax[0].set_xlabel('target')

ax[1].pie(vc.values, labels=[f"{i} ({v})" for i,v in zip(vc.index, vc.values)],
          autopct='%1.1f%%', colors=sns.color_palette('muted'), startangle=90)
ax[1].set_title('Répartition (%)')
plt.tight_layout()
plt.show()

# %% [markdown]
# 2) Histogrammes et densités des features (séparées par classe)

# %%
features = [c for c in df.columns if c != 'target']
n = len(features)
cols = 2
rows = int(np.ceil(n/cols))
plt.figure(figsize=(12, 4*rows))
for i, f in enumerate(features, 1):
    plt.subplot(rows, cols, i)
    sns.histplot(data=df, x=f, hue='target', kde=True, stat="density", common_norm=False, palette='tab10', alpha=0.5)
    plt.title(f)
plt.tight_layout()
plt.show()

# %% [markdown]
# 3) Boxplots pour repérer outliers et différences entre classes

# %%
plt.figure(figsize=(12, 4*rows))
for i, f in enumerate(features, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(x='target', y=f, data=df, palette='Set2')
    plt.title(f"Boxplot de {f} par target")
plt.tight_layout()
plt.show()

# %% [markdown]
# 4) Matrice de corrélation

# %%
corr = df[features].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='vlag', center=0)
plt.title("Corrélation entre features")
plt.show()

# %% [markdown]
# 5) Pairplot (utile si peu de features — peut être lent)

# %%
try:
    sns.pairplot(df[features + ['target']], hue='target', diag_kind='kde', corner=True, plot_kws={'alpha':0.6})
    plt.suptitle("Pairplot (features vs target)", y=1.02)
    plt.show()
except Exception as e:
    print("Pairplot skipped:", e)

# %% [markdown]
# 6) Résumé et suggestion automatique simple

# %%
minority_count = vc.min()
majority_count = vc.max()
imbalance_ratio = majority_count / minority_count if minority_count else np.inf
minority_pct = (minority_count / len(df)) * 100

print(f"Minority class count: {minority_count} ({minority_pct:.1f}%)")
print(f"Imbalance ratio (maj/min): {imbalance_ratio:.2f}")

if minority_pct < 40:
    print("Conclusion: jeu de données déséquilibré — considérer SMOTE / class_weight / échantillonnage.")
else:
    print("Conclusion: classes relativement équilibrées.")

# %% [markdown]
# Instructions:
# - Exécuter chaque cellule dans l'ordre.
# - Adapter `features` si vous voulez exclure des colonnes non pertinentes.
# - Pour analyse plus poussée (SHAP, calibration, tuning), je peux ajouter des cellules dédiées.
```# filepath: c:\Users\LENOVO T14s\Documents\learn\kaggle\spok\pcos_visual_analysis.py
# Jupyter-style script (VSCode interactive / Jupyter can run cells separated by "# %%")

# %% [markdown]
# Analyse visuelle PCOS — distribution des classes et graphiques exploratoires

# %%
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# tenter de réutiliser les utilitaires existants
try:
    from pcos_analysis import load_data, clean_columns
except Exception:
    # fallback: charger directement le CSV dans le même dossier
    DATA_PATH = os.path.join(os.path.dirname(__file__), "PCOS_infertility.csv")
    def load_data(path=DATA_PATH):
        return pd.read_csv(path)
    def clean_columns(df):
        return df.rename(columns=lambda c: c.strip())

# %% 
# Chargement et préparation légère
df = load_data()
df = clean_columns(df)

# convertir en numérique comme dans pcos_analysis
for col in df.columns:
    if col != 'target':
        df[col] = pd.to_numeric(df[col], errors='coerce')
df['target'] = pd.to_numeric(df['target'], errors='coerce').astype('Int64')

print("Shape:", df.shape)
display(df.head())

# %% [markdown]
# 1) Distribution de la cible — vérifier l'imbalancedness

# %%
# Distribution cible
vc = df['target'].value_counts(dropna=False)
pct = df['target'].value_counts(normalize=True) * 100
print("Counts:\n", vc.to_dict())
print("\nPercent:\n", pct.round(2).to_dict())

fig, ax = plt.subplots(1,2, figsize=(10,4))
sns.barplot(x=vc.index.astype(str), y=vc.values, ax=ax[0], palette='muted')
ax[0].set_title('Counts par classe (target)')
ax[0].set_xlabel('target')

ax[1].pie(vc.values, labels=[f"{i} ({v})" for i,v in zip(vc.index, vc.values)],
          autopct='%1.1f%%', colors=sns.color_palette('muted'), startangle=90)
ax[1].set_title('Répartition (%)')
plt.tight_layout()
plt.show()

# %% [markdown]
# 2) Histogrammes et densités des features (séparées par classe)

# %%
features = [c for c in df.columns if c != 'target']
n = len(features)
cols = 2
rows = int(np.ceil(n/cols))
plt.figure(figsize=(12, 4*rows))
for i, f in enumerate(features, 1):
    plt.subplot(rows, cols, i)
    sns.histplot(data=df, x=f, hue='target', kde=True, stat="density", common_norm=False, palette='tab10', alpha=0.5)
    plt.title(f)
plt.tight_layout()
plt.show()

# %% [markdown]
# 3) Boxplots pour repérer outliers et différences entre classes

# %%
plt.figure(figsize=(12, 4*rows))
for i, f in enumerate(features, 1):
    plt.subplot(rows, cols, i)
    sns.boxplot(x='target', y=f, data=df, palette='Set2')
    plt.title(f"Boxplot de {f} par target")
plt.tight_layout()
plt.show()

# %% [markdown]
# 4) Matrice de corrélation

# %%
corr = df[features].corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='vlag', center=0)
plt.title("Corrélation entre features")
plt.show()

# %% [markdown]
# 5) Pairplot (utile si peu de features — peut être lent)

# %%
try:
    sns.pairplot(df[features + ['target']], hue='target', diag_kind='kde', corner=True, plot_kws={'alpha':0.6})
    plt.suptitle("Pairplot (features vs target)", y=1.02)
    plt.show()
except Exception as e:
    print("Pairplot skipped:", e)

# %% [markdown]
# 6) Résumé et suggestion automatique simple

# %%
minority_count = vc.min()
majority_count = vc.max()
imbalance_ratio = majority_count / minority_count if minority_count else np.inf
minority_pct = (minority_count / len(df)) * 100

print(f"Minority class count: {minority_count} ({minority_pct:.1f}%)")
print(f"Imbalance ratio (maj/min): {imbalance_ratio:.2f}")

if minority_pct < 40:
    print("Conclusion: jeu de données déséquilibré — considérer SMOTE / class_weight / échantillonnage.")
else:
    print("Conclusion: classes relativement équilibrées.")

# %% [markdown]
# Instructions:
# - Exécuter chaque cellule dans l'ordre.
# - Adapter `features` si vous voulez exclure des colonnes non pertinentes.
# - Pour analyse plus poussée (SHAP, calibration, tuning), je peux ajouter des cellules dédiées.

SyntaxError: invalid syntax (264210962.py, line 131)